# Text Clustering dengan TF-IDF dan K-Means

Notebook ini digunakan untuk melakukan clustering pada dokumen teks.

Tahapan:
1. Import library
2. Load dataset
3. Text preprocessing
4. TF-IDF
5. Elbow Method
6. Silhouette Score
7. K-Means Clustering
8. PCA
9. Top terms setiap cluster
10. Confusion Matrix
11. Menyimpan hasil

Dataset dummy dapat menggunakan `dummy_text_clustering.csv`.

## 1. Import Library

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

## 2. Konfigurasi

`TEXT_COLUMN` berisi teks yang akan diproses dan `LABEL_COLUMN`
digunakan sebagai label asli untuk Confusion Matrix.

In [ ]:
TEXT_FILE = "dummy_text_clustering.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

K_MIN = 2
K_MAX = 8
SELECTED_K = None

RANDOM_STATE = 42

## 3. Stopwords Bahasa Indonesia

In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan",
    "pada", "adalah", "itu", "ini", "atau", "juga", "akan",
    "dalam", "tidak", "ada", "karena", "oleh", "sebagai",
    "dapat", "bagi", "lebih", "sangat", "sudah", "telah",
    "bahwa", "agar", "mereka", "kami", "kita", "saya",
    "anda", "dia", "ia", "sebuah", "para", "jadi", "seperti",
    "namun", "tetapi", "bisa", "masih", "hanya", "setelah",
    "sebelum", "saat", "hingga", "tanpa", "antara",
    "terhadap", "tentang", "menjadi", "merupakan", "secara"
}

## 4. Text Preprocessing

Tahap preprocessing meliputi lowercase, penghapusan URL,
mention, simbol hashtag, karakter non-huruf, dan stopwords.

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [
        word for word in text.split()
        if word not in STOPWORDS_ID
    ]

    return " ".join(tokens)

## 5. Load Dataset dan Preprocessing

Dataset harus memiliki kolom `text`. Kolom `label` digunakan
sebagai actual label jika tersedia.

In [ ]:
df = pd.read_csv(TEXT_FILE)

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Kolom '{TEXT_COLUMN}' tidak ditemukan.")

labels = df[LABEL_COLUMN] if LABEL_COLUMN in df.columns else None

df = df.dropna(subset=[TEXT_COLUMN]).copy()
df["clean_text"] = df[TEXT_COLUMN].apply(preprocess_text)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

if labels is not None:
    labels = df[LABEL_COLUMN]

print("Jumlah dokumen:", len(df))
display(df.head())

## 6. TF-IDF

TF-IDF mengubah teks menjadi representasi numerik berdasarkan
bobot kata pada setiap dokumen.

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),
    max_features=3000,
    sublinear_tf=True
)

X_tfidf = vectorizer.fit_transform(df["clean_text"])

print("Jumlah dokumen:", X_tfidf.shape[0])
print("Jumlah fitur TF-IDF:", X_tfidf.shape[1])

X = X_tfidf.toarray()
X_scaled = StandardScaler().fit_transform(X)

## 7. Elbow Method

In [ ]:
max_k = min(K_MAX, len(X_scaled) - 1)

if max_k < K_MIN:
    raise ValueError("Jumlah dokumen terlalu sedikit.")

k_values = range(K_MIN, max_k + 1)
inertias = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method - Text Clustering")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Silhouette Score

Silhouette Score dihitung untuk setiap kandidat K.
K dengan nilai tertinggi digunakan jika `SELECTED_K = None`.

In [ ]:
silhouette_results = {}

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    cluster_labels = model.fit_predict(X_scaled)
    silhouette_results[k] = silhouette_score(X_scaled, cluster_labels)

for k, score in silhouette_results.items():
    print(f"K={k}: {score:.4f}")

best_k = max(silhouette_results, key=silhouette_results.get)

if SELECTED_K is None:
    SELECTED_K = best_k

print(f"\nK yang digunakan: {SELECTED_K}")

## 9. Final K-Means

In [ ]:
kmeans = KMeans(
    n_clusters=SELECTED_K,
    random_state=RANDOM_STATE,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)
final_silhouette = silhouette_score(X_scaled, clusters)

print(f"Silhouette Score final: {final_silhouette:.4f}")

## 10. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters,
    cmap="tab10",
    s=50
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Text Clusters")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 11. Top Terms Setiap Cluster

Rata-rata TF-IDF digunakan untuk melihat kata yang paling
dominan pada masing-masing cluster.

In [ ]:
terms = vectorizer.get_feature_names_out()

for cluster_id in range(SELECTED_K):
    cluster_indices = np.where(clusters == cluster_id)[0]

    mean_tfidf = np.asarray(
        X_tfidf[cluster_indices].mean(axis=0)
    ).ravel()

    top_indices = mean_tfidf.argsort()[::-1][:10]
    top_terms = terms[top_indices]

    print(f"Cluster {cluster_id}:")
    print(", ".join(top_terms))
    print()

## 12. Jumlah Dokumen Setiap Cluster

In [ ]:
unique, counts = np.unique(clusters, return_counts=True)

for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} dokumen")

## 13. Confusion Matrix

Confusion Matrix membandingkan label asli dengan cluster hasil K-Means.
Karena nomor cluster tidak otomatis sama dengan nomor kelas, dilakukan
pemetaan cluster ke label menggunakan Hungarian Algorithm.

In [ ]:
def best_cluster_label_mapping(y_true, y_cluster):
    true_labels = np.unique(y_true)
    cluster_labels = np.unique(y_cluster)

    true_to_int = {label: i for i, label in enumerate(true_labels)}
    y_true_int = np.array([true_to_int[x] for x in y_true])

    n = max(len(true_labels), len(cluster_labels))
    contingency = np.zeros((n, n), dtype=int)

    for t, c in zip(y_true_int, y_cluster):
        contingency[t, c] += 1

    row_ind, col_ind = linear_sum_assignment(-contingency)

    mapping = {c: r for r, c in zip(row_ind, col_ind)}

    for c in cluster_labels:
        if c not in mapping:
            mapping[c] = 0

    y_pred_mapped = np.array([mapping[c] for c in y_cluster])

    return true_labels, y_true_int, y_pred_mapped


if labels is not None and not labels.isna().all():
    valid = ~labels.isna()

    y_true = labels[valid].astype(str).values
    y_cluster = np.array(clusters)[valid]

    true_labels, y_true_int, y_pred_mapped = best_cluster_label_mapping(
        y_true,
        y_cluster
    )

    cm = confusion_matrix(
        y_true_int,
        y_pred_mapped,
        labels=np.arange(len(true_labels))
    )

    accuracy = accuracy_score(y_true_int, y_pred_mapped)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=true_labels,
        yticklabels=true_labels
    )
    plt.xlabel("Predicted Cluster")
    plt.ylabel("Actual Label")
    plt.title(f"Confusion Matrix - Text Clustering\nAccuracy = {accuracy:.4f}")
    plt.tight_layout()
    plt.show()

    print(pd.DataFrame(cm, index=true_labels, columns=true_labels))
    print(f"\nAccuracy: {accuracy:.4f}")
else:
    print("Label asli tidak tersedia. Confusion Matrix tidak dapat dibuat.")

## Helper Interpretasi Otomatis

Cell berikut membaca hasil clustering, silhouette score, PCA, dan confusion matrix lalu memberikan interpretasi otomatis.

In [ ]:
# Helper Interpretasi Hasil Text Clustering

print("=" * 70)
print("HELPER INTERPRETASI TEXT CLUSTERING")
print("=" * 70)

# 1. Silhouette
print("\n1. INTERPRETASI SILHOUETTE SCORE")
print(f"Silhouette Score = {final_silhouette:.4f}")

if final_silhouette >= 0.70:
    print("Hasil: sangat baik.")
    print("Dokumen dalam cluster memiliki kemiripan yang kuat.")
elif final_silhouette >= 0.50:
    print("Hasil: cukup baik.")
    print("Dokumen relatif terpisah dengan cukup baik antar cluster.")
elif final_silhouette >= 0.25:
    print("Hasil: sedang/lemah.")
    print("Beberapa dokumen masih memiliki kemiripan dengan cluster lain.")
else:
    print("Hasil: kurang baik.")
    print("Pemisahan dokumen antar cluster masih lemah.")

# 2. Cluster size
print("\n2. DISTRIBUSI DOKUMEN")
cluster_counts = pd.Series(clusters).value_counts().sort_index()

for cluster_id, count in cluster_counts.items():
    percentage = count / len(clusters) * 100
    print(f"Cluster {cluster_id}: {count} dokumen ({percentage:.2f}%)")

# 3. Top terms
print("\n3. KARAKTERISTIK TOPIK SETIAP CLUSTER")

terms = vectorizer.get_feature_names_out()

for cluster_id in range(SELECTED_K):
    cluster_indices = np.where(clusters == cluster_id)[0]

    mean_tfidf = np.asarray(
        X_tfidf[cluster_indices].mean(axis=0)
    ).ravel()

    top_indices = mean_tfidf.argsort()[::-1][:5]
    top_terms = terms[top_indices]

    print(
        f"Cluster {cluster_id}: "
        + ", ".join(top_terms)
    )

# 4. Confusion Matrix
print("\n4. INTERPRETASI CONFUSION MATRIX")

if labels is not None and not labels.isna().all():
    print(f"Accuracy setelah mapping = {accuracy:.4f}")

    if accuracy >= 0.90:
        print("Hasil: sangat baik.")
        print("Sebagian besar dokumen berhasil dikelompokkan")
        print("sesuai dengan label aslinya.")
    elif accuracy >= 0.75:
        print("Hasil: baik.")
        print("Mayoritas dokumen sudah berada pada kelompok yang sesuai.")
    elif accuracy >= 0.60:
        print("Hasil: cukup.")
        print("Masih terdapat kesalahan pengelompokan antar kelas.")
    else:
        print("Hasil: kurang baik.")
        print("Banyak dokumen masih tertukar antar kelas.")

    print("\nKecocokan setiap label:")

    for i, label in enumerate(true_labels):
        row_total = cm[i].sum()
        correct = cm[i, i]
        pct = correct / row_total * 100 if row_total else 0

        print(
            f"- {label}: {correct}/{row_total} "
            f"benar ({pct:.2f}%)"
        )
else:
    print("Confusion Matrix tidak dapat dibuat karena label asli")
    print("tidak tersedia.")

# 5. PCA
print("\n5. INTERPRETASI PCA")

explained = pca.explained_variance_ratio_

print(
    f"PC1 menjelaskan {explained[0] * 100:.2f}% variasi data."
)
print(
    f"PC2 menjelaskan {explained[1] * 100:.2f}% variasi data."
)
print(
    f"Total PC1 + PC2 = {explained.sum() * 100:.2f}% variasi data."
)

print("\nKESIMPULAN TEXT CLUSTERING")
print(
    f"Dokumen berhasil dikelompokkan menjadi {SELECTED_K} cluster "
    f"menggunakan TF-IDF dan K-Means."
)
print(
    f"Silhouette Score akhir adalah {final_silhouette:.4f}."
)